In [2]:
import os, random, math, warnings
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
 
import torchvision.models as tvm
import torchvision.transforms.functional as TF
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from tqdm.auto import tqdm
 
warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True


/Users/rahulmac/Documents/Projects/projects/Image_Forensics_v2/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [4]:
DATA_ROOT = Path("./dataset")          # root of GenImage dataset
CKPT_DIR  = Path("./checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

In [5]:
CFG = dict(
    img_size      = 224,
    batch_size    = 32,
    num_workers   = 4,
    epochs        = 30,
    lr            = 2e-4,
    weight_decay  = 1e-2,
    label_smooth  = 0.05,
    warmup_epochs = 3,
    grad_clip     = 1.0,
    amp           = True,
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)
print(f"Device: {CFG['device']}  |  AMP: {CFG['amp']}")

Device: cpu  |  AMP: True


In [6]:
# ── Cell 2 · Forensic Dataset & DataLoader (albumentations ≥ 1.4 compatible) ──
#
# Directory layout expected:
#   DATA_ROOT/
#     train/  val/  test/
#       real_pool/   → class 0
#       gan_pool/    → class 1
#       mj_pool/     → class 1
#       sd_pool/     → class 1
#
# Fallback: If train/val structure doesn't exist, pools at root are split 80/20

import cv2
AI_POOLS  = ["gan_pool", "mj_pool", "sd_pool"]
REAL_POOL = "real_pool"

_SZ = (CFG["img_size"], CFG["img_size"])   # albumentations ≥ 1.4 wants a tuple

# ── Social-media degradation augmentations (training only) ─────────────────────
TRAIN_TRANSFORMS = A.Compose([
    A.RandomResizedCrop(size=_SZ, scale=(0.7, 1.0)),          # ← size= tuple
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ImageCompression(quality_range=(40, 90), p=1.0),    # ← quality_range tuple
        A.Downscale(scale_range=(0.5, 0.9), p=1.0),           # ← scale_range tuple
    ], p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.Sharpen(alpha=(0.1, 0.4), p=1.0),
    ], p=0.4),
    A.OneOf([
        A.GaussNoise(std_range=(0.02, 0.12), p=1.0),          # ← std_range replaces var_limit
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
    ], p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Resize(height=CFG["img_size"], width=CFG["img_size"]),  # height/width kwargs still valid
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class GenImageDataset(Dataset):
    EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    def __init__(self, root: Path, split: str = "train", transform=None):
        self.transform = transform
        self.samples: list[tuple[Path, int]] = []

        split_dir = root / split                 # e.g.  ./dataset/train/
        
        # Try structured layout first (train/val subdirs exist)
        if (split_dir / REAL_POOL).exists():
            # ─ Structured layout: train/val/test subdirs ─
            real_dir = split_dir / REAL_POOL
            for p in sorted(real_dir.rglob("*")):
                if p.suffix.lower() in self.EXTENSIONS:
                    self.samples.append((p, 0))

            for pool in AI_POOLS:
                ai_dir = split_dir / pool
                if not ai_dir.exists():
                    continue
                for p in sorted(ai_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        self.samples.append((p, 1))
        else:
            # ─ Fallback: pools at root level, split 80/20 by split arg ─
            all_samples = []
            
            # Collect all images
            real_dir = root / REAL_POOL
            if real_dir.exists():
                for p in sorted(real_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        all_samples.append((p, 0))

            for pool in AI_POOLS:
                ai_dir = root / pool
                if not ai_dir.exists():
                    continue
                for p in sorted(ai_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        all_samples.append((p, 1))
            
            # Split into train/val using deterministic split
            random.seed(SEED)  # Ensure reproducibility
            random.shuffle(all_samples)
            split_idx = int(len(all_samples) * 0.8)
            
            if split == "train":
                self.samples = all_samples[:split_idx]
            elif split == "val":
                self.samples = all_samples[split_idx:]
            else:  # test
                self.samples = all_samples[split_idx:]  # Use val split for test

        random.shuffle(self.samples)
        labels = [s[1] for s in self.samples]
        n0, n1 = labels.count(0), labels.count(1)
        print(f"[{split}] real={n0:,}  AI={n1:,}  total={len(self.samples):,}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # cv2 faster than PIL for large datasets
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


def make_loaders(root: Path, cfg: dict):
    train_ds = GenImageDataset(root, "train", TRAIN_TRANSFORMS)
    val_ds   = GenImageDataset(root, "val",   VAL_TRANSFORMS)

    # Weighted sampler → handles class imbalance without oversampling real images
    labels   = [s[1] for s in train_ds.samples]
    counts   = np.bincount(labels).astype(float)
    weights  = 1.0 / counts[labels]
    sampler  = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], sampler=sampler,
        num_workers=cfg["num_workers"], pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg["batch_size"] * 2, shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True,
    )
    return train_loader, val_loader


train_loader, val_loader = make_loaders(DATA_ROOT, CFG)


[train] real=4,816  AI=4,784  total=9,600
[val] real=1,184  AI=1,216  total=2,400


In [7]:
# ── Cell 3 · Dual-Stream Forensic CNN ─────────────────────────────────────────
#
# Architecture:
#   Stream A : Spatial (RGB)   : EfficientNet-B4 pretrained → 1792-d features
#   Stream B : Frequency (DCT) : Patch-DCT extraction → lightweight CNN → 512-d
#   Fusion                     : Cross-modal attention → MLP head
#
# Rationale:
#   AI generators (GANs, diffusion, MJ) leave distinctive spectral fingerprints
#   in mid-to-high DCT frequencies that are nearly invisible in RGB but are
#   robustly detectable in the frequency domain (F3Net, DCT-Net, FreqNet).
#   The dual-stream + cross-attention fusion is the dominant paradigm in
#   CVPR/ICCV 2023-24 forensic literature.

# ── Patch-DCT frequency extractor ─────────────────────────────────────────────
class PatchDCT(nn.Module):
    """Splits image into non-overlapping patches, applies 2-D DCT per patch,
    then stacks the coefficient planes as a multi-channel feature map."""

    def __init__(self, patch_size: int = 8, top_k: int = 32):
        super().__init__()
        self.p = patch_size
        self.k = top_k                          # keep k lowest-frequency coefficients
        self._build_dct_basis(patch_size)

    def _build_dct_basis(self, N: int):
        basis = torch.zeros(N, N, N, N)
        for u in range(N):
            for v in range(N):
                cu = math.sqrt(1 / N) if u == 0 else math.sqrt(2 / N)
                cv = math.sqrt(1 / N) if v == 0 else math.sqrt(2 / N)
                for x in range(N):
                    for y in range(N):
                        basis[u, v, x, y] = (
                            cu * cv
                            * math.cos(math.pi * (2 * x + 1) * u / (2 * N))
                            * math.cos(math.pi * (2 * y + 1) * v / (2 * N))
                        )
        # (N*N, 1, N, N) reuse as depthwise convolution weights
        kernel = basis.view(N * N, 1, N, N)
        self.register_buffer("kernel", kernel)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        p = self.p
        # depthwise DCT per channel
        x_flat = x.view(B * C, 1, H, W)
        dct    = F.conv2d(x_flat, self.kernel, stride=p)   # (B*C, p², H/p, W/p)
        _, _, Hf, Wf = dct.shape
        dct    = dct.view(B, C, p * p, Hf, Wf)
        # keep top-k zig-zag coefficients (lowest freq → richest AI fingerprint region)
        dct    = dct[:, :, :self.k, :, :]      # (B, C, k, Hf, Wf)
        dct    = dct.view(B, C * self.k, Hf, Wf)
        return dct                              # (B, C·k, H/p, W/p)


# ── Frequency-domain CNN stream ────────────────────────────────────────────────
class FreqStream(nn.Module):
    def __init__(self, in_channels: int = 3 * 32, out_dim: int = 512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 128, 3, padding=1), nn.GELU(), nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.GELU(), nn.BatchNorm2d(256),
            nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.GELU(), nn.BatchNorm2d(512),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.encoder(x).flatten(1))


# ── Cross-modal attention fusion ───────────────────────────────────────────────
class CrossModalAttention(nn.Module):
    """Spatial features attend to frequency features and vice-versa."""

    def __init__(self, dim_s: int, dim_f: int, hidden: int = 512):
        super().__init__()
        self.q_s = nn.Linear(dim_s, hidden)
        self.k_f = nn.Linear(dim_f, hidden)
        self.v_f = nn.Linear(dim_f, hidden)
        self.q_f = nn.Linear(dim_f, hidden)
        self.k_s = nn.Linear(dim_s, hidden)
        self.v_s = nn.Linear(dim_s, hidden)
        self.scale  = hidden ** -0.5
        self.norm_s = nn.LayerNorm(hidden)
        self.norm_f = nn.LayerNorm(hidden)

    def forward(self, fs: torch.Tensor, ff: torch.Tensor):
        # spatial queries frequency
        a_sf = (self.q_s(fs) * self.scale) @ self.k_f(ff).unsqueeze(-1)
        a_sf = torch.sigmoid(a_sf.squeeze(-1)) * self.v_f(ff)
        # frequency queries spatial
        a_fs = (self.q_f(ff) * self.scale) @ self.k_s(fs).unsqueeze(-1)
        a_fs = torch.sigmoid(a_fs.squeeze(-1)) * self.v_s(fs)
        return self.norm_s(a_sf), self.norm_f(a_fs)


# ── Full dual-stream model ─────────────────────────────────────────────────────
class DualStreamForensicNet(nn.Module):
    SPATIAL_DIM = 1792          # EfficientNet-B4 penultimate feature dim
    FREQ_DIM    = 512
    FUSION_DIM  = 512

    def __init__(self, num_classes: int = 1, patch_size: int = 8, top_k: int = 32):
        super().__init__()

        # ── Stream A: spatial (RGB) ────────────────────────────────────────────
        backbone           = tvm.efficientnet_b4(weights=tvm.EfficientNet_B4_Weights.DEFAULT)
        self.spatial_feat  = nn.Sequential(*list(backbone.children())[:-2])  # drop pool+head
        self.spatial_pool  = nn.AdaptiveAvgPool2d(1)
        self.spatial_proj  = nn.Linear(self.SPATIAL_DIM, self.FUSION_DIM)

        # ── Stream B: frequency (DCT) ──────────────────────────────────────────
        self.patch_dct    = PatchDCT(patch_size, top_k)
        self.freq_stream  = FreqStream(in_channels=3 * top_k, out_dim=self.FREQ_DIM)

        # ── Cross-modal attention fusion ───────────────────────────────────────
        self.fusion = CrossModalAttention(self.FUSION_DIM, self.FREQ_DIM, self.FUSION_DIM)

        # ── Classification head ────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(self.FUSION_DIM * 2, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # spatial stream
        fs  = self.spatial_pool(self.spatial_feat(x)).flatten(1)  # (B, 1792)
        fs  = self.spatial_proj(fs)                                # (B, 512)

        # frequency stream
        dct = self.patch_dct(x)
        ff  = self.freq_stream(dct)                                # (B, 512)

        # cross-modal attention → fuse
        fs_att, ff_att = self.fusion(fs, ff)
        fused = torch.cat([fs_att, ff_att], dim=1)                 # (B, 1024)

        return self.head(fused).squeeze(1)                         # (B,)  raw logit


model = DualStreamForensicNet().to(CFG["device"])
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {total/1e6:.1f}M  |  Trainable: {trainable/1e6:.1f}M")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /Users/rahulmac/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [01:12<00:00, 1.08MB/s]


Params: 22.2M  |  Trainable: 22.2M
